In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.express as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
def parse_lines(lines):
    data = []
    for  line in lines:
        x = json.loads(line)
        data.append(x)

    data = pd.DataFrame.from_dict(data, orient="columns")
    return data

In [ ]:
with open("./foo.txt") as f:
    lines = f.readlines()

data = parse_lines(lines)
data = data.set_index(["i", "j", "k", "l"])
data["wctpf"] = data["wctpf"] / 1000;
data["wctpf_label"] = data["wctpf"].map({0: "Fully Mitigated", 17: "Mitigated", 1000000: "Unmitigated"})
data["tpf"] = data["dt"] / data["df"]
data["avg_tpf"] = data["acc_t"] / data["acc_f"]
data["frozen"] = data["irq"].apply(lambda x: True if "Freeze" in x else False)
data

In [ ]:
tpf=data.reset_index()
tpf[["wctpf", "tpf"]].groupby("wctpf").mean()


# plt.box(tpf[tpf["k"] == 40][tpf["tpf"] < 11], x="tpf", color="wctpf", log_x=True)


plt.box(tpf[tpf["k"] == 40][tpf["tpf"] > 100][tpf["wctpf"] != 25], x="tpf", color="wctpf_label", labels={
    "tpf": "Instantanious Execution-Time per Fuel Unit [ns/fuel]",
    "wctpf_label": ""
}, width=800, height=400)



In [ ]:
avg_tpf=data.groupby(["i", "j", "k"]).tail(1).reset_index()
fig = plt.histogram(avg_tpf, x="avg_tpf", color="wctpf_label", histnorm='percent', nbins=200,
    labels={
        "avg_tpf": "Total Execution-Time per Fuel Unit [ns/fuel]",
        "wctpf_label": ""
        },width=800, height=400
)
fig.add_vline(x=25, line_dash="dot", annotation_text="Mitigation Target:<br>25 ns/fuel", annotation_position="top right")
fig.write_image("mitigation-hist.pdf")
fig

In [ ]:
x=data.reset_index()
fig = plt.line(x[x["k"] == 0], x="acc_f", y="avg_tpf", color="wctpf_label", labels={"acc_f": "Computational Progress [fuel]", "avg_tpf": "Avg. Execution-Time per Fuel Unit [ns/fuel]", "wctpf_label": ""}, width=800, height=400)
fig.add_hline(y=25, line_dash="dot", annotation_text="Mitigation Target:<br>25 ns/fuel", annotation_position="bottom right")
fig.write_image("mitigation-time.pdf")
fig

In [ ]:
x = data.reset_index()[["i", "k", "fuel", "timestamp", "acc_t"]]
x = x.groupby(["i", "k"]).agg(["first", "last"])
x["eff"] = x["acc_t"]["last"] / (x["timestamp"]["last"] - x["timestamp"]["first"])
x=x.groupby("i").mean()
plt.line(x=x["fuel"]["first"], y=x["eff"], labels={"x": "Fuel-Chunk Size", "y": "Efficiency"})

In [ ]:
x=data.reset_index()
fig=plt.line(x[(x["k"] == 0) & (x["wctpf"] == 25)], x="acc_f", y="frozen", color="wctpf_label", labels={"acc_f": "Computational Progress [fuel]", "frozen": "Is Mitigation Active", "wctpf_label": ""}, width=800, height=400).update_traces({"fill": "tozeroy", "line": {"color": "orangered"}})
fig.write_image("mitigation-active.pdf")
fig